# Monitoring NLP Models in Production
### Drug Review Sentiment Classification + Data Drift Detection with Evidently

## 0. Setup and Dependencies

In [ ]:
# Install Evidently if not already installed
# !pip install evidently -q

import io, zipfile, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
np.random.seed(42)

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# Evidently
try:
    from evidently.legacy.report import Report
    from evidently.legacy.metric_preset import (
        DataDriftPreset, ClassificationPreset, DataQualityPreset
    )
    from evidently.legacy.metrics import (
        DatasetDriftMetric, DatasetMissingValuesMetric,
        ClassificationQualityMetric
    )
    EVIDENTLY_LEGACY = True
except ImportError:
    from evidently.report import Report
    from evidently.metric_preset import (
        DataDriftPreset, ClassificationPreset, DataQualityPreset
    )
    EVIDENTLY_LEGACY = False

print(f'Evidently API: {"legacy" if EVIDENTLY_LEGACY else "current"}')
print('All libraries loaded.')

## 1. Dataset — Drug Reviews

In [ ]:
# ── Try loading the real UCI Drug Reviews dataset ─────────────
DATASET_URL = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
               '00462/drugsCom_raw.zip')

df_raw = None
try:
    r = requests.get(DATASET_URL, timeout=12)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        train_file = [f for f in z.namelist() if 'train' in f.lower()][0]
        with z.open(train_file) as f:
            df_raw = pd.read_csv(f, sep='\t')
    df_raw.columns = df_raw.columns.str.strip()
    df_raw = df_raw.rename(columns={'review':'review','rating':'rating'})
    # Positive = rating >= 7, Negative = rating <= 4
    df_raw = df_raw[df_raw['rating'].notna()]
    df_raw['label'] = (df_raw['rating'] >= 7).astype(int)
    df_raw = df_raw[df_raw['rating'].apply(lambda x: x >= 7 or x <= 4)]
    df_raw = df_raw[['review','label','condition']].dropna().rename(
        columns={'condition':'condition'})
    print(f'Real dataset loaded: {len(df_raw):,} rows')
except Exception as e:
    print(f'Real dataset unavailable ({e}). Generating synthetic dataset...')

In [ ]:
# ── Synthetic dataset (representative structure) ──────────────
if df_raw is None:
    np.random.seed(42)

    CONDITIONS = ['depression','anxiety','diabetes','hypertension',
                  'migraine','chronic pain','insomnia','arthritis',
                  'asthma','ADHD']

    POS = [
        'This medication worked great for my {c}. Highly recommend.',
        'Excellent drug, reduced my {c} significantly within days.',
        'Best treatment I have tried for {c}. No side effects whatsoever.',
        'Very effective for {c}, felt better after one week of use.',
        'Amazing results treating {c}. My doctor was very impressed.',
        'Life changing medication for my {c}. I feel like myself again.',
        'Finally something that works for {c}. Five stars.',
        'Incredible improvement in my {c}. Would strongly recommend.',
        'This drug transformed my {c} management. Excellent.',
        'Wonderful results for {c}. Side effects minimal and manageable.',
    ]
    NEG = [
        'Terrible side effects from this drug for my {c}. Avoid.',
        'Did not help my {c} at all. Complete waste of money.',
        'Made my {c} significantly worse. Stopped after three days.',
        'Horrible experience using this for {c}. Pain actually increased.',
        'No improvement in my {c} whatsoever. Very disappointed.',
        'Caused severe nausea and dizziness. {c} completely unchanged.',
        'Worst medication I have tried for {c}. Do not recommend.',
        'Dangerous side effects with no benefit for {c}.',
        'This drug ruined my quality of life while treating {c}.',
        'Stopped taking after one week. {c} got worse not better.',
    ]

    records = []
    for _ in range(5000):
        cond = np.random.choice(CONDITIONS)
        pos  = np.random.rand() > 0.38
        tmpl = np.random.choice(POS if pos else NEG)
        # Add length variation and noise
        txt  = tmpl.format(c=cond)
        if np.random.rand() > 0.6:
            extra = np.random.choice([
                ' My family noticed the difference too.',
                ' Will continue using.',
                ' Doctor agreed with my assessment.',
                ' Would not take again.',
                ' Side effects were unbearable.',
            ])
            txt += extra
        records.append({
            'review':    txt,
            'condition': cond,
            'label':     int(pos),
            'rating':    np.random.randint(7,11) if pos else np.random.randint(1,5),
        })

    df_raw = pd.DataFrame(records)
    print(f'Synthetic dataset: {len(df_raw):,} rows')

print(f"Label distribution:\n{df_raw['label'].value_counts().to_string()}")
df_raw.head()

In [ ]:
# ── Quick EDA ─────────────────────────────────────────────────
df_raw['review_len'] = df_raw['review'].str.len()
df_raw['word_count'] = df_raw['review'].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class balance
counts = df_raw['label'].value_counts()
axes[0].bar(['Negative (0)', 'Positive (1)'], counts.values,
            color=['#E84040', '#55A868'], edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, f'{v:,}\n({v/len(df_raw)*100:.1f}%)',
                 ha='center', fontweight='bold')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Review length distribution
for lbl, color in [(1,'#55A868'),(0,'#E84040')]:
    axes[1].hist(df_raw[df_raw['label']==lbl]['review_len'],
                 bins=30, alpha=0.55, color=color, edgecolor='white',
                 label=f'{"Positive" if lbl else "Negative"}')
axes[1].set_title('Review Length Distribution', fontweight='bold')
axes[1].set_xlabel('Characters')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Condition distribution
top_cond = df_raw['condition'].value_counts().head(8)
axes[2].barh(top_cond.index[::-1], top_cond.values[::-1],
             color='#4C72B0', edgecolor='white')
axes[2].set_title('Top Conditions', fontweight='bold')
axes[2].set_xlabel('Count')
axes[2].grid(axis='x', alpha=0.3)

plt.suptitle('Drug Reviews — Exploratory Data Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Build the Sentiment Classification Model

In [ ]:
# ── Train / test split ────────────────────────────────────────
df_train, df_test = train_test_split(
    df_raw, test_size=0.20, random_state=42, stratify=df_raw['label']
)
print(f'Train: {len(df_train):,}  |  Test: {len(df_test):,}')

# ── TF-IDF + Logistic Regression pipeline ─────────────────────
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words='english',
        min_df=2,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(max_iter=500, C=1.0,
                               class_weight='balanced', random_state=42))
])

pipeline.fit(df_train['review'], df_train['label'])
print('Model trained.')

# ── Evaluate on test set ──────────────────────────────────────
y_pred  = pipeline.predict(df_test['review'])
y_proba = pipeline.predict_proba(df_test['review'])[:, 1]

print(f'\nTest Accuracy : {accuracy_score(df_test["label"], y_pred):.4f}')
print(f'Test F1-Score : {f1_score(df_test["label"], y_pred):.4f}')
print()
print(classification_report(df_test['label'], y_pred,
                             target_names=['Negative','Positive']))

In [ ]:
# ── Confusion matrix + top weighted tokens ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm   = confusion_matrix(df_test['label'], y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Negative','Positive'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')

# Top positive and negative tokens
tfidf   = pipeline.named_steps['tfidf']
clf     = pipeline.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())
coef    = clf.coef_[0]
top_pos = pd.Series(coef, index=feature_names).nlargest(10)
top_neg = pd.Series(coef, index=feature_names).nsmallest(10)
top_tok = pd.concat([top_pos, top_neg]).sort_values()

colors_tok = ['#E84040' if v < 0 else '#55A868' for v in top_tok.values]
axes[1].barh(top_tok.index, top_tok.values, color=colors_tok, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top 10 Positive and Negative Tokens\n(TF-IDF Logistic Regression Coefficients)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Coefficient (log-odds of Positive sentiment)')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Add prediction columns to test set
df_test = df_test.copy()
df_test['prediction']       = y_pred
df_test['prediction_proba'] = y_proba.round(4)

## 3. Simulate Data Quality Issues

In [ ]:
# ── Extract TF-IDF feature vectors for Evidently monitoring ───
# Evidently works with tabular DataFrames; we use TF-IDF embeddings as features
tfidf_train = tfidf.transform(df_train['review'])
tfidf_test  = tfidf.transform(df_test['review'])

# Reduce to top 50 features for readability in drift reports
top50_idx = np.argsort(np.abs(coef))[-50:]
feat_names_50 = feature_names[top50_idx]

df_ref = pd.DataFrame(
    tfidf_train[:, top50_idx].toarray(),
    columns=feat_names_50
)
df_ref['label']             = df_train['label'].values
df_ref['prediction']        = pipeline.predict(df_train['review'])
df_ref['prediction_proba']  = pipeline.predict_proba(df_train['review'])[:, 1]
df_ref['review_len']        = df_train['review_len'].values
df_ref['word_count']        = df_train['word_count'].values

df_cur = pd.DataFrame(
    tfidf_test[:, top50_idx].toarray(),
    columns=feat_names_50
)
df_cur['label']             = df_test['label'].values
df_cur['prediction']        = df_test['prediction'].values
df_cur['prediction_proba']  = df_test['prediction_proba'].values
df_cur['review_len']        = df_test['review_len'].values
df_cur['word_count']        = df_test['word_count'].values

print(f'Reference (train) features shape : {df_ref.shape}')
print(f'Current   (test)  features shape : {df_cur.shape}')

In [ ]:
# ── Simulate 4 types of data quality issues on a production batch ─
df_drifted = df_cur.copy()

# Issue 1: Missing values (10% of review_len set to NaN)
missing_idx = np.random.choice(len(df_drifted), size=int(len(df_drifted)*0.10), replace=False)
df_drifted.loc[df_drifted.index[missing_idx], 'review_len'] = np.nan

# Issue 2: Vocabulary shift — multiply key positive tokens by 0 (zeros them out = drift)
pos_token_cols = [c for c in feat_names_50 if c in ['great','excellent','worked','effective']]
df_drifted[pos_token_cols] = df_drifted[pos_token_cols] * 0.05

# Issue 3: Review length shift (simulate shorter, lower-quality reviews in production)
df_drifted['review_len']  = df_drifted['review_len'].fillna(0) * 0.4
df_drifted['word_count']  = df_drifted['word_count']           * 0.4

# Issue 4: Label shift (production has more negative reviews — e.g., seasonal side-effect reports)
n_flip = int(len(df_drifted) * 0.18)
flip_idx = df_drifted[df_drifted['label']==1].index[:n_flip]
df_drifted.loc[flip_idx, 'label'] = 0

print('Data quality issues injected:')
print(f'  Missing values in review_len : {df_drifted["review_len"].isna().sum()}')
print(f'  Label distribution (drifted) : {df_drifted["label"].value_counts().to_dict()}')
print(f'  Avg review_len (original)    : {df_cur["review_len"].mean():.1f}')
print(f'  Avg review_len (drifted)     : {df_drifted["review_len"].mean():.1f}')

In [ ]:
# ── Visualise the simulated drift ────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Label distribution shift
orig_counts    = df_cur['label'].value_counts().sort_index()
drifted_counts = df_drifted['label'].value_counts().sort_index()
x = np.arange(2)
axes[0,0].bar(x-0.2, orig_counts.values,    0.35, label='Original', color='#4C72B0', edgecolor='white')
axes[0,0].bar(x+0.2, drifted_counts.values, 0.35, label='Drifted',  color='#E84040', edgecolor='white')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(['Negative','Positive'])
axes[0,0].set_title('Label Distribution Shift', fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(axis='y', alpha=0.3)

# 2. Review length distribution shift
axes[0,1].hist(df_cur['review_len'].dropna(), bins=30,
               alpha=0.55, color='#4C72B0', edgecolor='white', label='Original')
axes[0,1].hist(df_drifted['review_len'].dropna(), bins=30,
               alpha=0.55, color='#E84040', edgecolor='white', label='Drifted')
axes[0,1].set_title('Review Length Distribution Shift', fontweight='bold')
axes[0,1].set_xlabel('Characters')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Vocabulary feature shift (key tokens)
key_tokens = pos_token_cols[:4] if pos_token_cols else feat_names_50[:4]
orig_means    = df_cur[key_tokens].mean()
drifted_means = df_drifted[key_tokens].mean()
x2 = np.arange(len(key_tokens))
axes[1,0].bar(x2-0.2, orig_means.values,    0.35, label='Original', color='#4C72B0', edgecolor='white')
axes[1,0].bar(x2+0.2, drifted_means.values, 0.35, label='Drifted',  color='#E84040', edgecolor='white')
axes[1,0].set_xticks(x2)
axes[1,0].set_xticklabels(key_tokens, rotation=15)
axes[1,0].set_title('Vocabulary Feature Mean Shift', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(axis='y', alpha=0.3)

# 4. Missing values comparison
miss_orig    = df_cur.isnull().sum()
miss_drifted = df_drifted.isnull().sum()
miss_df = pd.DataFrame({'Original': miss_orig, 'Drifted': miss_drifted})
miss_df = miss_df[miss_df.sum(axis=1) > 0]
miss_df.plot(kind='bar', ax=axes[1,1], color=['#4C72B0','#E84040'], edgecolor='white', rot=0)
axes[1,1].set_title('Missing Values Comparison', fontweight='bold')
axes[1,1].set_ylabel('Count')
axes[1,1].grid(axis='y', alpha=0.3)

plt.suptitle('Simulated Data Quality Issues and Drift', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Monitor Model Quality Decay with Evidently

In [ ]:
# ── Manual performance comparison: original vs drifted ───────
scenarios = {
    'Original (clean test)': df_cur,
    'Drifted (production)':  df_drifted,
}

print(f'{"Scenario":<30} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-'*65)
perf_results = {}
for name, dset in scenarios.items():
    yt, yp = dset['label'], dset['prediction']
    row = {
        'Accuracy':  accuracy_score(yt, yp),
        'Precision': precision_score(yt, yp, zero_division=0),
        'Recall':    recall_score(yt, yp),
        'F1':        f1_score(yt, yp, zero_division=0),
    }
    perf_results[name] = row
    print(f'{name:<30} {row["Accuracy"]:>10.3f} {row["Precision"]:>10.3f} '
          f'{row["Recall"]:>10.3f} {row["F1"]:>10.3f}')

In [ ]:
# ── Evidently: Data Quality Report ───────────────────────────
quality_report = Report(metrics=[DataQualityPreset()])
quality_report.run(
    reference_data = df_ref[['review_len','word_count','label','prediction']],
    current_data   = df_drifted[['review_len','word_count','label','prediction']],
)
quality_report.save_html('/tmp/quality_report.html')
print('Data Quality Report saved to /tmp/quality_report.html')
print('(In Colab, use quality_report.show() to render inline)')

In [ ]:
# ── Evidently: Data Drift Report ─────────────────────────────
drift_cols = ['review_len','word_count'] + list(feat_names_50[:10])

drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(
    reference_data = df_ref[drift_cols],
    current_data   = df_drifted[drift_cols],
)
drift_report.save_html('/tmp/drift_report.html')

# Extract drift summary
try:
    drift_json  = drift_report.as_dict()
    drift_metrics = drift_json['metrics'][0]['result']
    n_drifted   = drift_metrics.get('number_of_drifted_columns', 'N/A')
    n_total     = drift_metrics.get('number_of_columns', len(drift_cols))
    share       = drift_metrics.get('share_of_drifted_columns', 'N/A')
    print(f'Drifted columns : {n_drifted} / {n_total}')
    print(f'Drift share     : {share}')
except Exception:
    print('Drift report saved. Open /tmp/drift_report.html to view.')

In [ ]:
# ── Evidently: Classification Quality Report ──────────────────
clf_df_ref = df_ref[['label','prediction','prediction_proba']].copy()
clf_df_ref.columns = ['target','prediction','prediction_proba']

clf_df_cur = df_drifted[['label','prediction','prediction_proba']].copy()
clf_df_cur.columns = ['target','prediction','prediction_proba']

clf_report = Report(metrics=[ClassificationPreset()])
clf_report.run(
    reference_data = clf_df_ref,
    current_data   = clf_df_cur,
)
clf_report.save_html('/tmp/classification_report.html')
print('Classification Quality Report saved to /tmp/classification_report.html')

## 5. Visualise Model Decay and Corrective Actions

In [ ]:
# ── Simulate rolling production windows to track metric decay ─
np.random.seed(0)
n_windows = 8
window_labels = [f'Week {i+1}' for i in range(n_windows)]

# Performance degrades gradually over production windows
decay_f1  = [0.88 - i*0.045 + np.random.randn()*0.008 for i in range(n_windows)]
decay_acc = [0.91 - i*0.035 + np.random.randn()*0.006 for i in range(n_windows)]
drift_share = [0.05 + i*0.08 + np.random.randn()*0.01 for i in range(n_windows)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# F1 decay
axes[0].plot(window_labels, decay_f1, 'o-', color='#4C72B0', linewidth=2.5, markersize=7)
axes[0].axhline(0.75, color='crimson', linestyle='--', linewidth=1.5, label='Alert threshold (0.75)')
axes[0].fill_between(window_labels, decay_f1,
                     alpha=0.12, color='#4C72B0')
axes[0].set_title('F1-Score Over Production Windows', fontsize=12, fontweight='bold')
axes[0].set_ylabel('F1-Score')
axes[0].set_ylim(0.4, 1.0)
axes[0].legend(fontsize=9)
axes[0].tick_params(axis='x', rotation=25)
axes[0].grid(True, alpha=0.3)

# Accuracy decay
axes[1].plot(window_labels, decay_acc, 's-', color='#55A868', linewidth=2.5, markersize=7)
axes[1].axhline(0.78, color='crimson', linestyle='--', linewidth=1.5, label='Alert threshold (0.78)')
axes[1].set_title('Accuracy Over Production Windows', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.5, 1.0)
axes[1].legend(fontsize=9)
axes[1].tick_params(axis='x', rotation=25)
axes[1].grid(True, alpha=0.3)

# Drift share
colors_drift = ['#E84040' if v > 0.40 else '#DD8452' if v > 0.20 else '#55A868'
                for v in drift_share]
axes[2].bar(window_labels, drift_share, color=colors_drift, edgecolor='white')
axes[2].axhline(0.20, color='#DD8452', linestyle='--', linewidth=1.5, label='Warning (20%)')
axes[2].axhline(0.40, color='#E84040', linestyle='--', linewidth=1.5, label='Alert (40%)')
axes[2].set_title('Share of Drifted Features', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Drift Share')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
axes[2].legend(fontsize=9)
axes[2].tick_params(axis='x', rotation=25)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Model Quality Monitoring Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Metric comparison: original vs drifted bar chart ─────────
metrics_names = list(perf_results['Original (clean test)'].keys())
orig_vals     = list(perf_results['Original (clean test)'].values())
drift_vals    = list(perf_results['Drifted (production)'].values())

x = np.arange(len(metrics_names))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x-w/2, orig_vals,  w, label='Original (clean test)', color='#4C72B0', edgecolor='white')
bars2 = ax.bar(x+w/2, drift_vals, w, label='Drifted (production)',  color='#E84040', edgecolor='white')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.set_ylim(0, 1.2)
ax.set_title('Performance Degradation: Original vs Drifted Production Data',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Corrective Strategies and Monitoring Summary

### Monitoring Challenges Encountered

| Challenge | Description | Impact |
|---|---|---|
| **Data drift** | Key sentiment tokens decreased in frequency in production (vocabulary shift) | Model receives different input distributions than it was trained on, degrading predictions |
| **Label drift** | Production data contains more negative reviews than training data | Class imbalance shifts, changing what the model needs to optimise for |
| **Missing values** | A fraction of `review_len` feature values absent in production | Feature-level completeness decreases; imputation strategy needed |
| **Review length shift** | Production reviews are shorter on average (lower quality user input) | Shorter reviews carry less signal; TF-IDF weights change |
| **Model quality decay** | F1-score drops progressively across weekly production windows | Model becomes increasingly unreliable without retraining |

---

### Solutions Implemented and Recommended

| Strategy | Implementation | When to apply |
|---|---|---|
| **Evidently Data Quality Report** | `DataQualityPreset` detects missing values and distribution anomalies automatically | Every production batch |
| **Evidently Drift Report** | `DataDriftPreset` runs statistical tests (KS, chi-squared) on each feature | Weekly or per deployment |
| **Evidently Classification Report** | `ClassificationPreset` monitors accuracy, F1, precision, recall vs reference | After ground-truth labels become available |
| **Alerting thresholds** | Trigger retraining when drift share > 20% or F1 < 0.75 | Automated CI/CD pipeline |
| **Periodic retraining** | Retrain the TF-IDF + LR pipeline on a rolling window of recent labelled data | Monthly or when drift is detected |
| **Active learning** | Flag low-confidence predictions for human annotation → add to next training batch | When labelling capacity exists |
| **Concept drift handling** | Use sample weights to upweight recent examples; rebuild vocabulary periodically | When language patterns evolve |
| **Shadow deployment** | Run updated model alongside current model to compare before promoting | Before every major model update |

In [ ]:
# ── Final summary printout ────────────────────────────────────
print('='*60)
print('  NLP MONITORING SUMMARY')
print('='*60)
for name, res in perf_results.items():
    print(f'\n  [{name}]')
    for metric, val in res.items():
        print(f'    {metric:<12}: {val:.4f}')

print('\n  Evidently reports generated:')
print('    /tmp/quality_report.html        (Data Quality)')
print('    /tmp/drift_report.html          (Data Drift)')
print('    /tmp/classification_report.html (Classification Quality)')
print()
print('  In Colab, view inline with: report.show()')